# Zomato Delivery Operations Analysis Using PySpark
## Stage 1: Dataset Inspection & Telemetry Audit

**Author:** Computer Engineering Mini-Project  
**Framework:** Apache Spark 3.5+ / 4.0 via PySpark  
**Dataset:** `Zomato Dataset.csv` (45,584 records, 20 operational columns)

---

### Objectives of Stage 1:
1. Initialize a local PySpark `SparkSession`.
2. Ingest the raw delivery operations dataset without schema loss.
3. Verify dimensions (rows, columns) and structural data types.
4. Quantify true missing values across all fields (accounting for string `'NaN'` literals and nulls).
5. Verify exact duplicates and assess uniqueness of primary identifiers.
6. Profile all categorical distributions (traffic, weather, vehicle, city, order type).
7. Detect numerical range anomalies (ratings, age, GPS coordinates, timestamps).

### Step 1: Initialize PySpark Session
Apache Spark uses a lazy-execution architecture. The `SparkSession` is the single unified entry point for all DataFrame and Spark SQL operations.

In [1]:
import os
import sys

# Add project root to sys.path
sys.path.append(os.path.abspath('..'))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

# Initialize PySpark in local mode
spark = (
    SparkSession.builder
    .appName('Stage1_DatasetInspection')
    .master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')
print(f'Spark Master: {spark.sparkContext.master}')
print(f'Spark Version: {spark.version}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/13 22:49:13 WARN Utils: Your hostname, Rajdeeps-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.35 instead (on interface en0)
26/10/13 22:49:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/13 22:49:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Master: local[*]
Spark Version: 4.2.0


### Step 2: Load the Raw Dataset
We read the CSV with `header=True` and `inferSchema=False`. In early exploration, keeping columns as strings prevents Spark from silently replacing unparseable values (such as text `'NaN'` or malformed decimals) with nulls before we can inspect them.

In [2]:
csv_path = '../data/raw/Zomato Dataset.csv' if os.path.exists('../data/raw/Zomato Dataset.csv') else '../Zomato Dataset.csv'
raw_df = spark.read.option('header', 'true').option('inferSchema', 'false').csv(csv_path)

total_rows = raw_df.count()
total_cols = len(raw_df.columns)
print(f'Total Records: {total_rows:,}')
print(f'Total Columns: {total_cols}')

Total Records: 45,584
Total Columns: 20


### Step 3: Inspect Columns and Data Types

In [3]:
raw_df.printSchema()
raw_df.show(5, truncate=False)

root
 |-- ID: string (nullable = true)
 |-- Delivery_person_ID: string (nullable = true)
 |-- Delivery_person_Age: string (nullable = true)
 |-- Delivery_person_Ratings: string (nullable = true)
 |-- Restaurant_latitude: string (nullable = true)
 |-- Restaurant_longitude: string (nullable = true)
 |-- Delivery_location_latitude: string (nullable = true)
 |-- Delivery_location_longitude: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Time_Orderd: string (nullable = true)
 |-- Time_Order_picked: string (nullable = true)
 |-- Weather_conditions: string (nullable = true)
 |-- Road_traffic_density: string (nullable = true)
 |-- Vehicle_condition: string (nullable = true)
 |-- Type_of_order: string (nullable = true)
 |-- Type_of_vehicle: string (nullable = true)
 |-- multiple_deliveries: string (nullable = true)
 |-- Festival: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Time_taken (min): string (nullable = true)

+------+------------------+-----

### Step 4: True Missing Value Breakdown
In real-world data pipelines, missing values frequently arrive in different representations:
- Standard Spark `NULL`
- Blank whitespace string `''`
- Text literal `'NaN'` (often exported from pandas or raw logs)

We calculate exact counts and percentages across all 20 columns:

In [4]:
import pandas as pd

missing_records = []
for c in raw_df.columns:
    cnt = raw_df.filter(
        F.col(c).isNull() | 
        (F.trim(F.col(c)) == '') | 
        (F.trim(F.col(c)) == 'NaN')
    ).count()
    pct = round((cnt / total_rows) * 100, 2)
    missing_records.append({'Column': c, 'Missing Count': cnt, 'Missing %': pct})

missing_df = pd.DataFrame(missing_records)
print(missing_df.to_string(index=False))

                     Column  Missing Count  Missing %
                         ID              0       0.00
         Delivery_person_ID              0       0.00
        Delivery_person_Age           1854       4.07
    Delivery_person_Ratings           1908       4.19
        Restaurant_latitude              0       0.00
       Restaurant_longitude              0       0.00
 Delivery_location_latitude              0       0.00
Delivery_location_longitude              0       0.00
                 Order_Date              0       0.00
                Time_Orderd           1731       3.80
          Time_Order_picked              0       0.00
         Weather_conditions            616       1.35
       Road_traffic_density            601       1.32
          Vehicle_condition              0       0.00
              Type_of_order              0       0.00
            Type_of_vehicle              0       0.00
        multiple_deliveries            993       2.18
                   Festival 

### Step 5: Duplicate and Identifier Validation
We evaluate:
1. Exact duplicate rows across all 20 columns.
2. Whether `ID` is a globally unique primary key.
3. How many unique delivery riders (`Delivery_person_ID`) exist, and their delivery frequency.

In [5]:
distinct_rows = raw_df.distinct().count()
print(f'Distinct Rows: {distinct_rows:,}')
print(f'Exact Duplicate Rows: {total_rows - distinct_rows:,}')

unique_ids = raw_df.select('ID').distinct().count()
print(f'Unique Order IDs: {unique_ids:,} (Repeated IDs: {total_rows - unique_ids})')

unique_riders = raw_df.select('Delivery_person_ID').distinct().count()
print(f'Unique Delivery Partners: {unique_riders:,}')

print('\nTop 5 Most Active Delivery Partners:')
raw_df.groupBy('Delivery_person_ID').count().orderBy(F.desc('count')).show(5, truncate=False)

Distinct Rows: 45,584
Exact Duplicate Rows: 0
Unique Order IDs: 45,584 (Repeated IDs: 0)


Unique Delivery Partners: 1,320

Top 5 Most Active Delivery Partners:
+------------------+-----+
|Delivery_person_ID|count|
+------------------+-----+
|PUNERES01DEL01    |67   |
|JAPRES11DEL02     |67   |
|VADRES08DEL02     |66   |
|JAPRES03DEL01     |66   |
|RANCHIRES02DEL01  |66   |
+------------------+-----+
only showing top 5 rows


### Step 6: Categorical Distributions
We inspect categorical factors affecting delivery operations.

In [6]:
cat_cols = [
    'Weather_conditions', 
    'Road_traffic_density', 
    'Vehicle_condition', 
    'Type_of_order', 
    'Type_of_vehicle', 
    'multiple_deliveries', 
    'Festival', 
    'City'
]

for c in cat_cols:
    print(f'=== Value Counts: {c} ===')
    raw_df.groupBy(c).count().orderBy(F.desc('count')).show(truncate=False)

=== Value Counts: Weather_conditions ===


+------------------+-----+
|Weather_conditions|count|
+------------------+-----+
|Fog               |7653 |
|Stormy            |7584 |
|Cloudy            |7533 |
|Sandstorms        |7494 |
|Windy             |7422 |
|Sunny             |7282 |
|NaN               |616  |
+------------------+-----+

=== Value Counts: Road_traffic_density ===
+--------------------+-----+
|Road_traffic_density|count|
+--------------------+-----+
|Low                 |15476|
|Jam                 |14139|
|Medium              |10945|
|High                |4423 |
|NaN                 |601  |
+--------------------+-----+

=== Value Counts: Vehicle_condition ===


+-----------------+-----+
|Vehicle_condition|count|
+-----------------+-----+
|2                |15031|
|1                |15028|
|0                |15005|
|3                |520  |
+-----------------+-----+

=== Value Counts: Type_of_order ===


+-------------+-----+
|Type_of_order|count|
+-------------+-----+
|Snack        |11530|
|Meal         |11456|
|Drinks       |11321|
|Buffet       |11277|
+-------------+-----+

=== Value Counts: Type_of_vehicle ===
+----------------+-----+
|Type_of_vehicle |count|
+----------------+-----+
|motorcycle      |26429|
|scooter         |15273|
|electric_scooter|3814 |
|bicycle         |68   |
+----------------+-----+

=== Value Counts: multiple_deliveries ===


+-------------------+-----+
|multiple_deliveries|count|
+-------------------+-----+
|1                  |28151|
|0                  |14094|
|2                  |1985 |
|NaN                |993  |
|3                  |361  |
+-------------------+-----+

=== Value Counts: Festival ===


+--------+-----+
|Festival|count|
+--------+-----+
|No      |44460|
|Yes     |896  |
|NaN     |228  |
+--------+-----+

=== Value Counts: City ===
+-------------+-----+
|City         |count|
+-------------+-----+
|Metropolitian|34087|
|Urban        |10133|
|NaN          |1200 |
|Semi-Urban   |164  |
+-------------+-----+



### Step 7: Numerical Ranges & Sanity Audit
We inspect summary metrics (min, max, mean, stddev) and flag edge cases:
- Delivery-person age (check for underage or extreme values)
- Delivery-person ratings (valid range [1.0, 5.0])
- Delivery duration `Time_taken (min)`
- Geographic coordinates (latitude/longitude checks)

In [7]:
# Clean NaN strings to NULL before casting so Spark summary stats work properly
num_df = (
    raw_df.withColumn('Age', F.when(F.col('Delivery_person_Age') == 'NaN', None).otherwise(F.col('Delivery_person_Age').cast(DoubleType())))
          .withColumn('Rating', F.when(F.col('Delivery_person_Ratings') == 'NaN', None).otherwise(F.col('Delivery_person_Ratings').cast(DoubleType())))
          .withColumn('Delivery_Time_Min', F.when(F.col('Time_taken (min)') == 'NaN', None).otherwise(F.col('Time_taken (min)').cast(DoubleType())))
          .withColumn('Rest_Lat', F.col('Restaurant_latitude').cast(DoubleType()))
          .withColumn('Rest_Lon', F.col('Restaurant_longitude').cast(DoubleType()))
          .withColumn('Del_Lat', F.col('Delivery_location_latitude').cast(DoubleType()))
          .withColumn('Del_Lon', F.col('Delivery_location_longitude').cast(DoubleType()))
)

num_df.select('Age', 'Rating', 'Delivery_Time_Min', 'Rest_Lat', 'Rest_Lon', 'Del_Lat', 'Del_Lon').describe().show()

# Anomaly Checks
print('--- Specific Anomaly Counts ---')
print('Ratings == 6.0 (Out of bounds):', num_df.filter(F.col('Rating') == 6.0).count())
print('Ages == 15 (Underage deliveries):', num_df.filter(F.col('Age') == 15.0).count())
print('Ages == 50:', num_df.filter(F.col('Age') == 50.0).count())
print('Coordinates at 0.0 (GPS sensor failure):', num_df.filter(
    (F.col('Rest_Lat') == 0.0) | (F.col('Rest_Lon') == 0.0) | 
    (F.col('Del_Lat') == 0.0) | (F.col('Del_Lon') == 0.0)
).count())
print('Negative Latitude records:', num_df.filter((F.col('Rest_Lat') < 0) | (F.col('Del_Lat') < 0)).count())
print('Negative Longitude records:', num_df.filter((F.col('Rest_Lon') < 0) | (F.col('Del_Lon') < 0)).count())

+-------+------------------+------------------+------------------+-----------------+------------------+------------------+-----------------+
|summary|               Age|            Rating| Delivery_Time_Min|         Rest_Lat|          Rest_Lon|           Del_Lat|          Del_Lon|
+-------+------------------+------------------+------------------+-----------------+------------------+------------------+-----------------+
|  count|             43730|             43676|             45584|            45584|             45584|             45584|            45584|
|   mean|29.566910587697233| 4.633773697225024|26.293962793962795|17.01794839638953| 70.22968394987328|17.465479840514313| 70.8441609078637|
| stddev| 5.815063854587498|0.3347437201745533|  9.38429773366794| 8.18567377987961|22.885574863999715| 7.335561662334391|21.12057827510355|
|    min|              15.0|               1.0|              10.0|       -30.905562|        -88.366217|              0.01|             0.01|
|    max|    

Ages == 50: 53
Coordinates at 0.0 (GPS sensor failure): 3640


Negative Latitude records: 431
Negative Longitude records: 162


### Step 8: Order Date & Timestamp Inconsistencies
Examining how timestamps and dates are stored:

In [8]:
# Date range audit
date_df = raw_df.withColumn('Parsed_Date', F.to_date(F.col('Order_Date'), 'dd-MM-yyyy'))
date_df.select(
    F.min('Parsed_Date').alias('earliest_date'),
    F.max('Parsed_Date').alias('latest_date'),
    F.countDistinct('Parsed_Date').alias('distinct_days')
).show()

# Timestamp irregularities
excel_decimal_orders = raw_df.filter(F.col('Time_Orderd').contains('.')).count()
excel_decimal_picked = raw_df.filter(F.col('Time_Order_picked').contains('.')).count()
hhmmss_picked = raw_df.filter(F.col('Time_Order_picked').rlike('^[0-9]{2}:[0-9]{2}:[0-9]{2}')).count()

print(f'Time_Orderd with Excel decimal fractions: {excel_decimal_orders:,}')
print(f'Time_Order_picked with Excel decimal fractions: {excel_decimal_picked:,}')
print(f'Time_Order_picked formatted as HH:MM:SS: {hhmmss_picked:,}')

+-------------+-----------+-------------+
|earliest_date|latest_date|distinct_days|
+-------------+-----------+-------------+
|   2022-02-11| 2022-04-06|           44|
+-------------+-----------+-------------+



Time_Orderd with Excel decimal fractions: 3,638
Time_Order_picked with Excel decimal fractions: 3,687
Time_Order_picked formatted as HH:MM:SS: 880


### Step 9: Shutdown Spark Session

In [9]:
spark.stop()
print('SparkSession successfully stopped.')

SparkSession successfully stopped.
